# ClauseAI - Industry-Grade Contract Analysis

![ClauseAI](../images/ClauseAI_logo.jpeg)

## Overview

ClauseAI is a multi-agent contract analysis system built with **LangGraph + OpenAI**. It automates contract review with the rigor of a multidisciplinary legal team: classification, entity extraction, financial term extraction, clause retrieval, missing-clause detection, conflict detection, role-based review, compliance checks, risk scoring, and PII detection.

## What's New (v2 - Industry Grade)

| Capability | v1 | v2 |
| --- | --- | --- |
| LLM provider | OpenAI | OpenAI (configurable model) |
| Vector store | Pinecone (required) | **FAISS (default) + Pinecone optional** |
| Entity extraction (parties, dates, amounts) | none | **Full structured extractor** |
| Financial term extraction | none | **Amounts, currencies, payment terms, penalties** |
| Obligations extraction | none | **Per-party obligation list** |
| Missing-clause detection | none | **Diff against expected clause library** |
| Conflict detection between clauses | none | **Cross-section consistency check** |
| Compliance checks | none | **GDPR / CCPA / HIPAA / SOX configurable** |
| Risk scoring (Critical/High/Medium/Low) | none | **Per-finding + aggregate score** |
| PII detection | none | **Sensitive-data flagger** |
| Confidence scores | none | **0-1 score on every finding** |
| Retry logic + error handling | minimal | **Tenacity, structured errors** |
| Token & cost tracking | none | **OpenAI callback tracker** |
| Final report | plain text | **Structured JSON + Markdown + DOCX** |

---

## Architecture

```
                       ┌── classify_contract
                       │
                       ├── extract_entities (parties, dates, amounts, obligations)
                       │
                       ├── detect_pii
                       │
  contract_text ──────┼── retrieve_clauses (FAISS) ──fan-out──→ check_clause (parallel)
                       │
                       ├── missing_clause_check
                       │
                       ├── detect_conflicts
                       │
                       ├── compliance_check (GDPR/CCPA/HIPAA/SOX)
                       │
                       ├── create_review_plan ──fan-out──→ execute_role_review (parallel)
                       │
                       └── risk_scorer → generate_final_report
```

All fan-out branches use **LangGraph `Send` (map-reduce)** for parallel execution.

## 1. Install dependencies

In [ ]:
%%capture --no-stderr
%pip install --quiet -U \
    langgraph langchain-core langchain-openai langchain-community \
    faiss-cpu tiktoken tenacity \
    pydantic python-docx PyPDF2 ipywidgets

## 2. Environment & Configuration

Set `OPENAI_API_KEY` in your environment or `.env`. Pinecone is **optional** — the notebook defaults to local FAISS so you can run it standalone.

In [ ]:
import os, getpass, logging
from dataclasses import dataclass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

@dataclass
class Settings:
    openai_model: str = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
    openai_model_strong: str = os.environ.get("OPENAI_MODEL_STRONG", "gpt-4o")
    embedding_model: str = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
    temperature: float = float(os.environ.get("LLM_TEMPERATURE", "0"))
    vector_backend: str = os.environ.get("VECTOR_BACKEND", "faiss")  # 'faiss' or 'pinecone'
    clauses_path: str = os.environ.get("CLAUSES_PATH", "data/clauses.json")
    max_parallel: int = int(os.environ.get("MAX_PARALLEL", "6"))
    enable_compliance_checks: tuple = ("GDPR", "CCPA")

settings = Settings()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
log = logging.getLogger("clauseai")
log.info(f"Settings: model={settings.openai_model}, vector_backend={settings.vector_backend}")

## 3. Imports

In [ ]:
import json, re, operator, time
from datetime import datetime
from typing import List, Dict, Optional, Annotated, Literal, Any
from typing_extensions import TypedDict

from pydantic import BaseModel, Field, ConfigDict
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.callbacks import get_openai_callback

from langgraph.graph import StateGraph, START, END
from langgraph.constants import Send
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model=settings.openai_model, temperature=settings.temperature)
llm_strong = ChatOpenAI(model=settings.openai_model_strong, temperature=settings.temperature)
embeddings = OpenAIEmbeddings(model=settings.embedding_model)

## 4. Document Parsing

Robust loaders for PDF, DOCX, and plain text with page/section markers preserved.

In [ ]:
import PyPDF2
import docx as docx_lib

def load_pdf(path: str) -> str:
    text_parts = []
    with open(path, "rb") as fh:
        reader = PyPDF2.PdfReader(fh)
        for i, page in enumerate(reader.pages):
            text_parts.append(f"\n[Page {i+1}]\n{page.extract_text() or ''}")
    return "\n".join(text_parts).strip()

def load_docx(path: str) -> str:
    doc = docx_lib.Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

def load_contract(path: str) -> str:
    p = path.lower()
    if p.endswith(".pdf"): return load_pdf(path)
    if p.endswith(".docx"): return load_docx(path)
    if p.endswith(".txt") or p.endswith(".md"):
        with open(path, "r", encoding="utf-8") as fh:
            return fh.read()
    raise ValueError(f"Unsupported file type: {path}")

SECTION_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)[\.\)\s]+(.*)$", re.MULTILINE)

def split_into_sections(text: str) -> List[Dict]:
    """Split a contract into numbered sections (e.g., '1. Position', '2.1 Compensation')."""
    matches = list(SECTION_RE.finditer(text))
    if not matches:
        return [{"id": "0", "title": "Full Document", "text": text}]
    sections = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        sections.append({
            "id": m.group(1),
            "title": m.group(2).strip()[:120],
            "text": text[start:end].strip(),
        })
    return sections

## 5. Sample Contract

We embed a realistic sample contract so the notebook is fully runnable without external files.

In [ ]:
SAMPLE_CONTRACT = """EMPLOYMENT AGREEMENT

This Employment Agreement ("Agreement") is entered into as of January 15, 2026, by and between
Acme Robotics, Inc., a Delaware corporation with offices at 100 Market St, San Francisco, CA 94105 ("Employer"),
and Jane Doe, residing at 22 Oak Ave, San Francisco, CA 94110, SSN 123-45-6789 ("Employee").

1. Position and Duties
1.1 Employee shall serve as Senior Software Engineer reporting to the VP of Engineering.
1.2 Employee shall devote her full business time and attention to the duties of her position.

2. Compensation
2.1 Base Salary: Employer shall pay Employee an annual base salary of $185,000, payable bi-weekly.
2.2 Bonus: Employee may receive a discretionary annual bonus targeted at 15% of base salary.
2.3 Equity: Employee shall receive 8,000 RSUs vesting over four years, 25% after one year and the remainder monthly.

3. Term and Termination
3.1 Employment is at-will. Either party may terminate at any time, for any reason, without notice.
3.2 Upon termination, Employee shall receive accrued but unpaid wages.

4. Non-Compete
4.1 For a period of three (3) years following termination, Employee shall not, anywhere in the world,
engage in any business that competes with Employer in any capacity whatsoever.

5. Intellectual Property
5.1 All inventions, works, and developments created by Employee during employment, whether or not related
to Employer's business, and whether created on or off Employer premises, shall be the sole property of Employer.

6. Confidentiality
6.1 Employee shall hold Employer's confidential information in strict confidence indefinitely.

7. Data Processing
7.1 Employer may collect, store, and process Employee personal data including biometric data and health records
for any business purpose, and may share such data with third parties at its discretion.

8. Governing Law
8.1 This Agreement shall be governed by the laws of the State of California.

9. Entire Agreement
9.1 This Agreement constitutes the entire agreement between the parties.

Signed:
Acme Robotics, Inc.  ____________________   Jane Doe ____________________
"""

print(f"Sample contract: {len(SAMPLE_CONTRACT)} chars, {len(split_into_sections(SAMPLE_CONTRACT))} sections")

## 6. Clause Library + Vector Retriever (FAISS default)

We load the curated clause library (`data/clauses.json`) into FAISS for similarity retrieval. The retriever supports filter-by-contract-type and similarity search. A Pinecone backend is available behind an env flag for production multi-tenant deployments.

In [ ]:
class ClauseRetriever:
    """FAISS-backed clause retriever with metadata filtering."""

    def __init__(self, json_path: str):
        with open(json_path, "r", encoding="utf-8") as fh:
            self.library = json.load(fh)
        self.docs: List[Document] = []
        for ct in self.library:
            for cl in ct["clauses"]:
                content = f"Contract Type: {ct['contract_type']}\nClause: {cl['clause_title']}\n\n{cl['clause_text']}"
                self.docs.append(Document(
                    page_content=content,
                    metadata={
                        "contract_type": ct["contract_type"],
                        "clause_title": cl["clause_title"],
                        "clause_text": cl["clause_text"],
                        **cl.get("metadata", {}),
                    }
                ))
        log.info(f"Loaded {len(self.docs)} clauses across {len(self.library)} contract types")
        self.store = FAISS.from_documents(self.docs, embeddings)

    def by_contract_type(self, contract_type: str, k: int = 20) -> List[Dict]:
        return [
            {**d.metadata, "score": 1.0}
            for d in self.docs if d.metadata["contract_type"].lower() == contract_type.lower()
        ][:k]

    def similarity(self, query: str, contract_type: Optional[str] = None, k: int = 5) -> List[Dict]:
        results = self.store.similarity_search_with_score(query, k=k*3 if contract_type else k)
        out = []
        for doc, score in results:
            if contract_type and doc.metadata["contract_type"].lower() not in {
                contract_type.lower(), "general clauses"
            }:
                continue
            out.append({**doc.metadata, "score": float(score)})
            if len(out) >= k:
                break
        return out

    def expected_clause_titles(self, contract_type: str) -> List[str]:
        general = [d.metadata["clause_title"] for d in self.docs
                   if d.metadata["contract_type"] == "General Clauses"]
        specific = [d.metadata["clause_title"] for d in self.docs
                    if d.metadata["contract_type"].lower() == contract_type.lower()]
        return sorted(set(general + specific))

clause_retriever = ClauseRetriever(settings.clauses_path)

## 7. State Schemas (Pydantic)

Strongly-typed schemas for every node so we get structured outputs and clean validation.

In [ ]:
RiskLevel = Literal["Critical", "High", "Medium", "Low", "Info"]
RiskCategory = Literal[
    "Legal", "Financial", "IP", "Compliance", "Operational",
    "Privacy", "Termination", "Liability", "Other"
]

class ContractInfo(BaseModel):
    contract_type: str = Field(description="E.g. Employment Contract, NDA, SaaS Agreement")
    industry: Optional[str] = Field(default=None)
    governing_law: Optional[str] = None
    effective_date: Optional[str] = None
    parties: List[str] = Field(default_factory=list)
    summary: str = Field(default="", description="One-paragraph executive summary")

class Party(BaseModel):
    name: str
    role: str = Field(description="e.g. Employer, Employee, Licensor, Licensee")
    address: Optional[str] = None
    entity_type: Optional[str] = None

class FinancialTerm(BaseModel):
    label: str
    amount: Optional[str] = None
    currency: Optional[str] = "USD"
    cadence: Optional[str] = Field(default=None, description="one-time, monthly, annual, etc.")
    notes: Optional[str] = None

class KeyDate(BaseModel):
    label: str
    date_text: str
    iso_date: Optional[str] = None

class Obligation(BaseModel):
    party: str
    obligation: str
    deadline: Optional[str] = None

class EntityExtraction(BaseModel):
    parties: List[Party] = Field(default_factory=list)
    financial_terms: List[FinancialTerm] = Field(default_factory=list)
    key_dates: List[KeyDate] = Field(default_factory=list)
    obligations: List[Obligation] = Field(default_factory=list)

class Modification(BaseModel):
    original_text: str
    suggested_text: str
    reason: str
    risk_level: RiskLevel = "Medium"
    confidence: float = Field(default=0.7, ge=0, le=1)

class RiskFinding(BaseModel):
    title: str
    description: str
    category: RiskCategory
    risk_level: RiskLevel
    section_reference: Optional[str] = None
    recommendation: str = ""
    confidence: float = Field(default=0.7, ge=0, le=1)

class ComplianceFinding(BaseModel):
    framework: str  # e.g. GDPR, CCPA
    requirement: str
    status: Literal["Compliant", "Partial", "Non-Compliant", "Not Applicable"]
    explanation: str
    recommendation: Optional[str] = None

class MissingClause(BaseModel):
    clause_title: str
    importance: RiskLevel
    why_missing_matters: str
    suggested_text: str

class Conflict(BaseModel):
    section_a: str
    section_b: str
    description: str
    risk_level: RiskLevel
    resolution: str

class PIIFinding(BaseModel):
    type: str  # e.g. SSN, Bank Account, Health Data
    excerpt: str
    recommendation: str

class StepAnalysis(BaseModel):
    role: str = ""
    analysis: str
    modifications: List[Modification] = Field(default_factory=list)
    risk_findings: List[RiskFinding] = Field(default_factory=list)

class ReviewPlan(BaseModel):
    roles: List[str] = Field(description="Distinct legal/business roles for review")

class ContractReviewState(TypedDict, total=False):
    contract_text: str
    primary_objective: str
    specific_focus: Optional[str]
    sections_split: List[Dict]
    contract_info: ContractInfo
    entities: EntityExtraction
    pii_findings: List[PIIFinding]
    expected_clauses: List[str]
    retrieved_clauses: List[Dict]
    clause_check_results: Annotated[List[Dict], operator.add]
    missing_clauses: List[MissingClause]
    conflicts: List[Conflict]
    compliance_findings: List[ComplianceFinding]
    review_plan: ReviewPlan
    role_analyses: Annotated[List[StepAnalysis], operator.add]
    modifications: Annotated[List[Modification], operator.add]
    risk_findings: Annotated[List[RiskFinding], operator.add]
    overall_risk_score: float
    overall_risk_level: RiskLevel
    final_report_md: str
    final_report_json: Dict
    token_usage: Dict
    errors: Annotated[List[str], operator.add]

## 8. Robust LLM call helper

Wraps `llm.with_structured_output` with retries, timeouts, and structured fallback so a single transient failure can't kill the run.

In [ ]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(min=1, max=10),
    reraise=True,
)
def structured_call(model: ChatOpenAI, schema, system: str, human: str):
    return model.with_structured_output(schema).invoke([
        SystemMessage(content=system),
        HumanMessage(content=human),
    ])

def safe_structured_call(model, schema, system, human, default):
    try:
        return structured_call(model, schema, system, human)
    except Exception as e:
        log.warning(f"structured_call failed for {schema.__name__}: {e}")
        return default

## 9. Graph Nodes

Each node is small, single-responsibility, and returns a partial state update.

In [ ]:
def node_classify(state):
    info = safe_structured_call(
        llm, ContractInfo,
        "You are a senior contracts attorney. Identify contract type, industry, governing law, effective date, parties (just names), and write a one-paragraph executive summary.",
        f"Contract:\n{state['contract_text']}",
        ContractInfo(contract_type="Unknown", summary=""),
    )
    sections = split_into_sections(state["contract_text"])
    return {"contract_info": info, "sections_split": sections}

def node_extract_entities(state):
    entities = safe_structured_call(
        llm, EntityExtraction,
        """Extract structured entities from the contract:
- parties (name, role, address, entity_type)
- financial_terms (label, amount, currency, cadence, notes) - include salaries, bonuses, equity counts, fees, penalties, late fees
- key_dates (label, date_text, iso_date if derivable)
- obligations (party, obligation text, deadline if any)
Return only what's actually in the text.""",
        f"Contract:\n{state['contract_text']}",
        EntityExtraction(),
    )
    return {"entities": entities}

def node_detect_pii(state):
    class PIIList(BaseModel):
        items: List[PIIFinding] = Field(default_factory=list)
    result = safe_structured_call(
        llm, PIIList,
        """Identify any sensitive personal data appearing inline in the contract that should typically be redacted or stored separately. Examples: SSN, government IDs, bank accounts, health/biometric data, exact home addresses of natural persons. For each, give a redaction or handling recommendation.""",
        state["contract_text"],
        PIIList(items=[]),
    )
    return {"pii_findings": result.items}

def node_retrieve_clauses(state):
    contract_type = state["contract_info"].contract_type
    expected = clause_retriever.expected_clause_titles(contract_type)
    retrieved = clause_retriever.by_contract_type("General Clauses", k=20) + \
                clause_retriever.by_contract_type(contract_type, k=20)
    log.info(f"Retrieved {len(retrieved)} reference clauses for type={contract_type}")
    return {"retrieved_clauses": retrieved, "expected_clauses": expected}

def node_check_clause(state):
    clause = state["retrieved_clauses"][0] if isinstance(state.get("retrieved_clauses"), list) else state.get("clause")
    if not clause:
        return {"clause_check_results": []}
    result = safe_structured_call(
        llm, StepAnalysis,
        f"""You are a clause-clarity analyst. Given the reference clause below, evaluate whether the contract
adequately addresses the same concept. If yes, note that. If absent, weak, or ambiguous, suggest concrete
modifications and risk findings (with risk_level).

Reference clause:
Title: {clause.get('clause_title')}
Text: {clause.get('clause_text')}""",
        state["contract_text"],
        StepAnalysis(role=clause.get("clause_title", "Clause"), analysis="(no analysis)"),
    )
    result.role = clause.get("clause_title", "Clause")
    return {
        "clause_check_results": [{"clause_title": result.role, "analysis": result.analysis}],
        "modifications": result.modifications,
        "risk_findings": result.risk_findings,
    }

def node_missing_clauses(state):
    class MissingList(BaseModel):
        items: List[MissingClause] = Field(default_factory=list)
    expected = state.get("expected_clauses", [])
    result = safe_structured_call(
        llm, MissingList,
        f"""For a {state['contract_info'].contract_type}, the following clauses are typically expected:
{json.dumps(expected)}

Identify which of these expected clauses are MISSING or only weakly addressed. For each, explain why it matters and propose suggested clause text.""",
        state["contract_text"],
        MissingList(items=[]),
    )
    return {"missing_clauses": result.items}

def node_detect_conflicts(state):
    class ConflictList(BaseModel):
        items: List[Conflict] = Field(default_factory=list)
    sections_blob = "\n\n".join(
        f"[Section {s['id']}] {s['title']}\n{s['text']}" for s in state.get("sections_split", [])
    )
    result = safe_structured_call(
        llm, ConflictList,
        "Identify internal inconsistencies or conflicts between sections of the contract (e.g. notice periods, termination rights, IP ownership scopes, governing-law mismatches). For each, identify the conflicting sections, explain the conflict, assign a risk level, and propose a resolution.",
        sections_blob,
        ConflictList(items=[]),
    )
    return {"conflicts": result.items}

def node_compliance(state):
    class ComplianceList(BaseModel):
        items: List[ComplianceFinding] = Field(default_factory=list)
    frameworks = list(settings.enable_compliance_checks)
    result = safe_structured_call(
        llm_strong, ComplianceList,
        f"""Evaluate the contract against the following frameworks: {frameworks}.
For each framework, list the most material requirements and whether the contract is Compliant / Partial / Non-Compliant / Not Applicable, with a short explanation and recommendation.""",
        state["contract_text"],
        ComplianceList(items=[]),
    )
    return {"compliance_findings": result.items}

def node_review_plan(state):
    info = state["contract_info"]
    plan = safe_structured_call(
        llm, ReviewPlan,
        f"""Build a list of 4-6 distinct legal/business roles that should each review a {info.contract_type}
in the {info.industry or 'general'} industry. Examples: Employment Law Counsel, IP Counsel, Compliance Officer,
Financial Terms Analyst, Risk Manager, Data Privacy Officer. Choose roles that best fit this specific contract.""",
        f"Primary objective: {state.get('primary_objective','')}\nSpecific focus: {state.get('specific_focus','')}",
        ReviewPlan(roles=["Generalist Legal Counsel"]),
    )
    return {"review_plan": plan}

def node_role_review(state):
    role = state.get("role") or (state["review_plan"].roles[0] if state.get("review_plan") else "Counsel")
    result = safe_structured_call(
        llm, StepAnalysis,
        f"""You are a {role}. Review this contract from your professional perspective. Output:
- analysis: detailed findings
- modifications: precise text changes (original_text must be exact substrings of the contract; risk_level set)
- risk_findings: discrete risk items with category and risk_level""",
        state["contract_text"],
        StepAnalysis(role=role, analysis=""),
    )
    result.role = role
    return {
        "role_analyses": [result],
        "modifications": result.modifications,
        "risk_findings": result.risk_findings,
    }

RISK_WEIGHTS = {"Critical": 1.0, "High": 0.7, "Medium": 0.4, "Low": 0.15, "Info": 0.05}

def aggregate_risk(findings: List[RiskFinding]) -> tuple[float, RiskLevel]:
    if not findings:
        return 0.0, "Low"
    score = sum(RISK_WEIGHTS.get(f.risk_level, 0.4) * f.confidence for f in findings) / max(len(findings), 1)
    level: RiskLevel = (
        "Critical" if score >= 0.75 else
        "High" if score >= 0.55 else
        "Medium" if score >= 0.3 else
        "Low"
    )
    return round(score, 3), level

def node_aggregate_risk(state):
    score, level = aggregate_risk(state.get("risk_findings", []))
    return {"overall_risk_score": score, "overall_risk_level": level}

## 10. Final Report Generator

Produces both a structured JSON report and a markdown executive summary.

In [ ]:
def node_final_report(state):
    info: ContractInfo = state["contract_info"]
    entities: EntityExtraction = state.get("entities", EntityExtraction())
    risks: List[RiskFinding] = state.get("risk_findings", [])
    mods: List[Modification] = state.get("modifications", [])
    compliance: List[ComplianceFinding] = state.get("compliance_findings", [])
    missing: List[MissingClause] = state.get("missing_clauses", [])
    conflicts: List[Conflict] = state.get("conflicts", [])
    pii: List[PIIFinding] = state.get("pii_findings", [])
    role_analyses: List[StepAnalysis] = state.get("role_analyses", [])

    report_json = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "contract_info": info.model_dump(),
        "primary_objective": state.get("primary_objective", ""),
        "specific_focus": state.get("specific_focus", ""),
        "overall_risk": {
            "score": state.get("overall_risk_score"),
            "level": state.get("overall_risk_level"),
        },
        "entities": entities.model_dump(),
        "risk_findings": [r.model_dump() for r in risks],
        "modifications": [m.model_dump() for m in mods],
        "compliance": [c.model_dump() for c in compliance],
        "missing_clauses": [m.model_dump() for m in missing],
        "conflicts": [c.model_dump() for c in conflicts],
        "pii": [p.model_dump() for p in pii],
        "role_analyses": [r.model_dump() for r in role_analyses],
    }

    def fmt_list(items, fmt):
        return "\n".join(fmt(i) for i in items) if items else "_None identified._"

    md = []
    md.append(f"# Contract Review Report\n")
    md.append(f"**Generated:** {report_json['generated_at']}\n")
    md.append(f"## Executive Summary\n")
    md.append(f"- **Type:** {info.contract_type}")
    md.append(f"- **Industry:** {info.industry or 'N/A'}")
    md.append(f"- **Governing Law:** {info.governing_law or 'N/A'}")
    md.append(f"- **Effective Date:** {info.effective_date or 'N/A'}")
    md.append(f"- **Parties:** {', '.join(info.parties) if info.parties else 'N/A'}")
    md.append(f"- **Overall Risk:** **{state.get('overall_risk_level','?')}** (score: {state.get('overall_risk_score','?')})\n")
    md.append(f"\n{info.summary}\n")

    md.append("## Risk Findings\n")
    md.append(fmt_list(
        sorted(risks, key=lambda r: -RISK_WEIGHTS.get(r.risk_level, 0)),
        lambda r: f"- **[{r.risk_level} | {r.category}]** {r.title} — {r.description}  _Recommendation: {r.recommendation}_",
    ))

    md.append("\n## Suggested Modifications\n")
    md.append(fmt_list(
        sorted(mods, key=lambda m: -RISK_WEIGHTS.get(m.risk_level, 0)),
        lambda m: f"- **[{m.risk_level}]** _{m.original_text[:120]}…_ → _{m.suggested_text[:120]}…_  ({m.reason})",
    ))

    md.append("\n## Missing Clauses\n")
    md.append(fmt_list(
        missing,
        lambda m: f"- **[{m.importance}]** {m.clause_title} — {m.why_missing_matters}",
    ))

    md.append("\n## Conflicts & Inconsistencies\n")
    md.append(fmt_list(
        conflicts,
        lambda c: f"- **[{c.risk_level}]** {c.section_a} ↔ {c.section_b}: {c.description}  _Resolution: {c.resolution}_",
    ))

    md.append("\n## Compliance\n")
    md.append(fmt_list(
        compliance,
        lambda c: f"- **{c.framework}** — {c.requirement}: **{c.status}**. {c.explanation}",
    ))

    md.append("\n## PII / Sensitive Data\n")
    md.append(fmt_list(
        pii,
        lambda p: f"- **{p.type}**: '{p.excerpt[:80]}…' — {p.recommendation}",
    ))

    md.append("\n## Role-Based Analyses\n")
    for r in role_analyses:
        md.append(f"### {r.role}\n{r.analysis}\n")

    return {"final_report_md": "\n".join(md), "final_report_json": report_json}

## 11. Map-Reduce dispatchers

In [ ]:
def fanout_clauses(state):
    return [
        Send("check_clause", {"contract_text": state["contract_text"], "clause": c})
        for c in state["retrieved_clauses"][: max(8, len(state["retrieved_clauses"]))]
    ]

def fanout_roles(state):
    return [
        Send("role_review", {"contract_text": state["contract_text"], "role": role})
        for role in state["review_plan"].roles
    ]

## 12. Build the Graph

In [ ]:
builder = StateGraph(ContractReviewState)

builder.add_node("classify", node_classify)
builder.add_node("extract_entities", node_extract_entities)
builder.add_node("detect_pii", node_detect_pii)
builder.add_node("retrieve_clauses", node_retrieve_clauses)
builder.add_node("check_clause", node_check_clause)
builder.add_node("missing_clauses", node_missing_clauses)
builder.add_node("detect_conflicts", node_detect_conflicts)
builder.add_node("compliance", node_compliance)
builder.add_node("review_plan", node_review_plan)
builder.add_node("role_review", node_role_review)
builder.add_node("aggregate_risk", node_aggregate_risk)
builder.add_node("final_report", node_final_report)

# After classification, fan out to several independent analyzers in parallel
builder.add_edge(START, "classify")
for nxt in ["extract_entities", "detect_pii", "retrieve_clauses",
            "missing_clauses", "detect_conflicts", "compliance", "review_plan"]:
    builder.add_edge("classify", nxt)

# Map-reduce branches
builder.add_conditional_edges("retrieve_clauses", fanout_clauses, ["check_clause"])
builder.add_conditional_edges("review_plan", fanout_roles, ["role_review"])

# Reducers feed risk aggregation
for src in ["extract_entities", "detect_pii", "check_clause", "missing_clauses",
            "detect_conflicts", "compliance", "role_review"]:
    builder.add_edge(src, "aggregate_risk")

builder.add_edge("aggregate_risk", "final_report")
builder.add_edge("final_report", END)

graph = builder.compile(checkpointer=MemorySaver())
log.info("Graph compiled")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print("Could not render mermaid graph:", e)

## 13. Run the analysis with token tracking

In [ ]:
input_state = {
    "contract_text": SAMPLE_CONTRACT,
    "primary_objective": "Identify high-risk clauses, ensure GDPR/CCPA compliance, and propose negotiation points.",
    "specific_focus": "Non-compete enforceability, IP scope, data processing, PII handling.",
}

config = {"configurable": {"thread_id": f"run-{int(time.time())}"}}

with get_openai_callback() as cb:
    t0 = time.time()
    result = graph.invoke(input_state, config=config)
    elapsed = time.time() - t0

print(f"\nElapsed: {elapsed:.1f}s")
print(f"Tokens: prompt={cb.prompt_tokens}, completion={cb.completion_tokens}, total={cb.total_tokens}")
print(f"Cost: ${cb.total_cost:.4f}")
print(f"Risk: {result.get('overall_risk_level')} (score={result.get('overall_risk_score')})")

In [ ]:
from IPython.display import Markdown
Markdown(result["final_report_md"])

In [ ]:
# Save full structured report as JSON for downstream tools / dashboards
with open("contract_report.json", "w", encoding="utf-8") as fh:
    json.dump(result["final_report_json"], fh, indent=2, default=str)
with open("contract_report.md", "w", encoding="utf-8") as fh:
    fh.write(result["final_report_md"])
print("Saved contract_report.json and contract_report.md")

## 14. Generate redlined DOCX

In [ ]:
from docx import Document
from docx.shared import RGBColor
from docx.oxml import OxmlElement

def add_strikethrough(run):
    rPr = run._element.get_or_add_rPr()
    rPr.append(OxmlElement("w:strike"))

def export_redlined_docx(contract_text: str, mods: List[Modification], out_path: str):
    doc = Document()
    doc.add_heading("Redlined Contract", level=1)
    paragraphs = contract_text.split("\n")
    for para in paragraphs:
        modified = False
        for mod in mods:
            if mod.original_text and mod.original_text.strip() and mod.original_text in para:
                modified = True
                p = doc.add_paragraph()
                run_orig = p.add_run(mod.original_text)
                add_strikethrough(run_orig)
                run_new = p.add_run("\n" + mod.suggested_text)
                run_new.font.color.rgb = RGBColor(0xC0, 0x00, 0x00)
                run_reason = p.add_run(f"\n[{mod.risk_level}] {mod.reason}")
                run_reason.italic = True
                run_reason.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
                break
        if not modified:
            doc.add_paragraph(para)
    doc.save(out_path)
    return out_path

out = export_redlined_docx(result["contract_text"], result.get("modifications", []), "Modified_Contract.docx")
print("Saved", out)

## 15. Next: Streamlit App

See [`streamlit_app/`](./streamlit_app/) for a production-ready UI built around the same LangGraph workflow:

- Multi-page UI: Upload → Live Analysis → Risk Dashboard → Detailed Report → Redline → Q&A Chat → Export
- OpenAI as LLM provider
- Centralized prompts in `streamlit_app/prompts/prompts.py`
- Streaming progress with per-node status indicators
- Cost & latency telemetry
- DOCX / PDF / JSON / Markdown export

```bash
cd streamlit_app
pip install -r requirements.txt
streamlit run app.py
```